# ESG-Talk Score Exploration

Per-transcript cosine similarities between each call's TF-IDF vector and the E, S, G reference corpus vectors.

| Variable | Definition |
|---|---|
| `e_talk` | $\cos(\vec{t},\, \vec{c}_E)$ |
| `s_talk` | $\cos(\vec{t},\, \vec{c}_S)$ |
| `g_talk` | $\cos(\vec{t},\, \vec{c}_G)$ |

Computed for three segment sets: **combined** (pres + answers), **pres** (presentation only), **answers** (Q&A answers only).

Source: `data/processed/esg_talk.csv`

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_columns", 20)

PROJECT_ROOT = Path("..").resolve()
PROC_DIR     = PROJECT_ROOT / "data" / "processed"
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
FIG_DIR      = PROJECT_ROOT / "reports" / "figures"
TAB_DIR      = PROJECT_ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

SETS    = ("combined", "pres", "answers")
CORPORA = ("e", "s", "g")
CORPUS_LABELS = {"e": "Environment", "s": "Social", "g": "Governance"}
COLORS = {"e": "#2ca02c", "s": "#1f77b4", "g": "#ff7f0e"}

# Score columns grouped by set
SCORE_COLS = {s: [f"{c}_talk_{s}" for c in CORPORA] for s in SETS}
ALL_SCORE_COLS = [col for cols in SCORE_COLS.values() for col in cols]

print("Config ready.")

## Load data

Load `esg_talk.csv` and merge with the call-level metadata (date, ticker, company name) from `list_earnings_calls_group_project_upload.csv`.

In [ ]:
# ESG talk scores
df_talk = pd.read_csv(PROC_DIR / "esg_talk.csv", dtype={"transcript_id": str})

# Call metadata
df_meta = pd.read_csv(RAW_DIR / "list_earnings_calls_group_project_upload.csv",
                      dtype=str)
df_meta["transcript_id"] = df_meta["filename"].str.replace(r"\.txt$", "", regex=True)
df_meta["year_call"]     = df_meta["year_call"].astype(int)

# Merge
df = df_talk.merge(
    df_meta[["transcript_id", "comnam", "gvkey", "permno",
             "year_call", "month_call", "day_call", "date_call"]],
    on="transcript_id", how="left"
)

matched = df["comnam"].notna().sum()
print(f"Rows: {len(df):,}   Metadata matched: {matched:,}   Unmatched: {len(df)-matched:,}")
print(f"\nYear range: {df['year_call'].min()} – {df['year_call'].max()}")
df[ALL_SCORE_COLS].describe()

## Score distributions

Histograms for e_talk, s_talk, g_talk (combined set). Scores are right-skewed — most calls have very low ESG content, with a long tail of high-ESG calls.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, corp in zip(axes, CORPORA):
    col  = f"{corp}_talk_combined"
    vals = df[col]
    # clip top 1% for readability
    cap  = vals.quantile(0.99)
    ax.hist(vals.clip(upper=cap), bins=80, color=COLORS[corp],
            edgecolor="white", linewidth=0.3)
    ax.set_title(f"{CORPUS_LABELS[corp]}\n({col})", fontsize=12)
    ax.set_xlabel("Cosine similarity")
    ax.set_ylabel("# transcripts")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
    ax.text(0.97, 0.95, f"median={vals.median():.5f}\nmean={vals.mean():.5f}\nmax={vals.max():.4f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

fig.suptitle("ESG-Talk Score Distributions (combined, top 1% clipped)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "esg_talk_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/esg_talk_distributions.png")

## Presentation vs. answers: score comparison

How much does ESG content differ between the prepared presentation and the Q&A answers?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cap_q = 0.995   # clip both axes at 99.5th percentile for readability

for ax, corp in zip(axes, CORPORA):
    x = df[f"{corp}_talk_pres"]
    y = df[f"{corp}_talk_answers"]
    xc, yc = x.quantile(cap_q), y.quantile(cap_q)
    ax.scatter(x.clip(upper=xc), y.clip(upper=yc),
               alpha=0.15, s=4, color=COLORS[corp], linewidths=0)
    lim = max(xc, yc)
    ax.plot([0, lim], [0, lim], "k--", lw=0.8, label="pres = answers")
    corr = x.corr(y)
    ax.set_title(f"{CORPUS_LABELS[corp]}\nr = {corr:.3f}", fontsize=12)
    ax.set_xlabel("pres score")
    ax.set_ylabel("answers score")
    ax.legend(fontsize=8)

fig.suptitle("Presentation vs. Answers ESG-Talk Scores", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "esg_talk_pres_vs_answers.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/esg_talk_pres_vs_answers.png")

## Correlation matrix across all 9 scores

Pearson correlations between all combinations of E/S/G × combined/pres/answers.

In [ ]:
corr = df[ALL_SCORE_COLS].corr()

# Shorten labels for display
short = {c: c.replace("_talk_combined", "_cmb")
              .replace("_talk_pres",     "_pres")
              .replace("_talk_answers",  "_ans")
         for c in ALL_SCORE_COLS}
corr_display = corr.rename(index=short, columns=short)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_display, dtype=bool), k=1)
sns.heatmap(corr_display, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=-1, vmax=1, mask=mask, ax=ax,
            linewidths=0.5, annot_kws={"size": 9})
ax.set_title("Pearson Correlations — ESG-Talk Scores", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "esg_talk_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

# Also save as table
corr.to_csv(TAB_DIR / "esg_talk_correlation.csv")
print("Saved → reports/figures/esg_talk_correlation.png")
print("Saved → reports/tables/esg_talk_correlation.csv")

## Time trend: mean ESG-talk scores by year

Average score per calendar year for each dimension (combined set).

In [ ]:
annual = (df.groupby("year_call")[SCORE_COLS["combined"]]
            .mean()
            .reset_index())

fig, ax = plt.subplots(figsize=(12, 5))
for corp in CORPORA:
    col = f"{corp}_talk_combined"
    ax.plot(annual["year_call"], annual[col],
            marker="o", markersize=4, label=CORPUS_LABELS[corp],
            color=COLORS[corp])

ax.set_title("Mean ESG-Talk Score by Year (combined segment)", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Mean cosine similarity")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "esg_talk_time_trend.png", dpi=150, bbox_inches="tight")
plt.show()

# Also display as table
annual.set_index("year_call").rename(columns={
    f"{c}_talk_combined": CORPUS_LABELS[c] for c in CORPORA
})

## Top and bottom transcripts

Which calls score highest and lowest on each dimension?

In [ ]:
DISPLAY_COLS = ["transcript_id", "comnam", "year_call",
                "e_talk_combined", "s_talk_combined", "g_talk_combined"]
TOP_N = 10

for corp in CORPORA:
    col = f"{corp}_talk_combined"
    print(f"\n{'='*60}")
    print(f"  Top-{TOP_N}  {CORPUS_LABELS[corp]} (e_talk = {col})")
    print(f"{'='*60}")
    display(
        df.nlargest(TOP_N, col)[DISPLAY_COLS]
          .reset_index(drop=True)
          .style.format({c: "{:.6f}" for c in DISPLAY_COLS if "talk" in c})
    )

## Summary statistics table

Descriptive stats for all 9 scores, saved to `reports/tables/`.

In [ ]:
stats = df[ALL_SCORE_COLS].describe().T
stats.index = stats.index.str.replace("_talk_", " | ").str.upper()
stats.to_csv(TAB_DIR / "esg_talk_summary_stats.csv")
print("Saved → reports/tables/esg_talk_summary_stats.csv\n")
display(stats.style.format("{:.6f}"))